In [1]:
import os
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, f1_score, balanced_accuracy_score
from utils.logistic_lasso_regression_fista import LogisticLassoRegressionFISTA
from utils.missing_schemas import MCAR, MNAR, MAR1, MAR2
from sklearn.neighbors import KNeighborsClassifier
from utils.unlabeled_log_reg import generate_missing_label_data, unlabeled_log_reg_approach_1, naive_approach, run_cv_averaged

DATA_DIR = "../data/.ipynb_checkpoints/"
SPAM_DIR = "spambase"
COLLEGE_DIR = "college"
ELECTRICITY_DIR = "electricity"
EMPLOYEE_DIR = "employee"

In [2]:
results = pd.DataFrame()

In [2]:
datasets = [SPAM_DIR, COLLEGE_DIR, ELECTRICITY_DIR, EMPLOYEE_DIR]
schemas = ["MCAR", "MAR1", "MAR2", "MNAR"]

# Approach 1 - kNN with soft labels

### one split

In [ ]:
n_neighbors = 10
missing_rate = 0.2

for dataset in datasets:
    print(dataset)
    X = pd.read_csv(os.path.join(DATA_DIR, dataset, "X.csv"))
    y = pd.read_csv(os.path.join(DATA_DIR, dataset, "Y.csv")).iloc[:, 0]
    missing_label_data = generate_missing_label_data(X, y, missing_rate)
    for schema in schemas:    
        print(schema)
        row = unlabeled_log_reg_approach_1(missing_label_data[schema]["X_train"], missing_label_data[schema]["y_train"], 
                             missing_label_data["test"]["X_test"], missing_label_data["test"]["y_test"],
                            n_neighbors = n_neighbors, dataset_name = dataset, missing_schema = schema, label_type = 'proba')
        results = pd.concat([results, row], ignore_index=True)

spambase
MCAR
MAR1
MAR2
MNAR
college
MCAR


### crossvalidation

In [ ]:
missing_rate = 0.2

results = pd.DataFrame()

for dataset in datasets:
    print(dataset)
    X = pd.read_csv(os.path.join(DATA_DIR, dataset, "X.csv"))
    y = pd.read_csv(os.path.join(DATA_DIR, dataset, "Y.csv")).iloc[:, 0]
    df = run_cv_averaged(X, y, dataset, missing_rate)
    print(df)
    results = pd.concat([results, df], axis=0)

spambase
oracle method


In [ ]:
print(results)

# Approach 2 - naive modeling

In [ ]:
missing_rate = 0.2

for dataset in datasets:
    X = pd.read_csv(os.path.join(DATA_DIR, dataset, "X.csv"))
    y = pd.read_csv(os.path.join(DATA_DIR, dataset, "Y.csv")).iloc[:, 0]
    missing_label_data = generate_missing_label_data(X, y, missing_rate)
    for schema in schemas:    
        row = naive_approach(missing_label_data[schema]["X_train"], missing_label_data[schema]["y_train"], 
                             missing_label_data["test"]["X_test"], missing_label_data["test"]["y_test"], 
                             dataset_name = dataset,  missing_schema = schema)
        results = pd.concat([results, row], ignore_index=True)

In [ ]:
print(results)

In [4]:
row = naive_approach(missing_label_data["MCAR"]["X_train"], missing_label_data["MCAR"]["y_train"], 
                             missing_label_data["test"]["X_test"], missing_label_data["test"]["y_test"], dataset_name = 'spam',  missing_schema = "MCAR" )

results = pd.concat([results, row], ignore_index=True)